In [1]:
import pandas as pd
import numpy as np
from collections import Counter

In [2]:
data = pd.read_csv(r"../../data/processed/data_vif.csv")

x = data["Risk_Label"].astype(str).str.strip().str.lower()

data["Risk_Label"] = x.map({
    "lowrisk": 0,
    "low risk": 0,
    "low": 0,
    "0": 0,
    "highrisk": 1,
    "high risk": 1,
    "high": 1,
    "1": 1
})

print(data["Risk_Label"].value_counts(dropna=False))

data["Risk_Label"] = data["Risk_Label"].astype(int)

data.head()

Risk_Label
0    3672
1     436
Name: count, dtype: int64


,Date,KODEX 200_Premium,KOSDAQ_return(%),NASDAQ_return(%),Brent Crude Oil_return(%),Gold Spot_return(%),USD/KRW_change(%),KOSPI 200 Volume,return(%),KOSPI 200 lagged_1_return(%),...,GJR_VaR_5_t1,Signal1_Buy,Signal1_Sell,Signal2_Buy,Signal2_Sell,Signal3_Buy,Signal3_Sell,Signal4_Buy,Signal4_Sell,Risk_Label
0,2009-04-17,-0.20,-2.796416,0.157320,0.0,-1.362586,-0.549095,251600000.0,-0.233192,0.303254,...,-2.736404,0,0,0,0,0,0,0,0,0
1,2009-04-20,-0.34,1.668519,-3.953850,0.0,2.234472,0.128139,183300000.0,0.564563,-0.233192,...,-2.640948,0,0,0,0,0,0,0,0,0
2,2009-04-21,-0.14,1.061549,2.191930,0.0,-0.553958,1.998728,204200000.0,-0.197523,0.564563,...,-2.553087,0,0,0,0,0,0,0,0,0
3,2009-04-22,-0.19,2.524236,0.137996,0.0,1.093648,-0.570187,291400000.0,1.408955,-0.197523,...,-2.469280,0,0,0,0,0,0,0,0,0
4,2009-04-23,-0.17,0.818378,0.369276,0.0,1.568707,-0.970084,277400000.0,0.992765,1.408955,...,-2.386329,0,0,0,0,0,1,0,0,0


In [ ]:
wrap=['NASDAQ_return(%)', 'KOSPI 200_PPO', 'KOSPI 200_OG', 'KOSDAQ_return(%)', 'KOSPI 200 Volume', 'Gold Spot_return(%)', 'USD/KRW_change(%)', 'KOSPI 200 lagged_2_return(%)', 'Signal4_Buy', 'Signal4_Sell', 'VKOSPI_Change(%)', 'Signal3_Buy', 'Signal3_Sell', 'Signal1_Buy', 'Signal1_Sell']

mul=['NASDAQ_return(%)', 'Brent Crude Oil_return(%)', 'Gold Spot_return(%)', 'KOSPI 200 lagged_1_return(%)', 'VKOSPI_Change(%)', 'KOSPI 200_BB_width', 'KOSPI 200_ADX14', 'return(%)', 'KOSDAQ_return(%)', 'KOSPI 200_PPO']

boruta=['KOSPI 200 lagged_1_return(%)', 'KOSPI 200 Volume', 'KOSPI 200_PPO_Hist', 'KOSPI 200_DMI14', 'GJR_VaR_5_t1', 'KOSPI 200_OG', 'Brent Crude Oil_return(%)', 'VKOSPI_Close', 'NASDAQ_return(%)']

In [4]:

# ============================================================
# #1. 하드보팅 함수
# ============================================================

def hard_voting_feature_selection(
    wrap_features,
    mul_features,
    boruta_features,
    target_col='Risk_Label',
    vote_threshold=2
):

    selection_dict = {
        'wrap': wrap_features,
        'mul': mul_features,
        'boruta': boruta_features
    }

    # 타겟 제거 + 중복 제거
    cleaned = {}
    for method, features in selection_dict.items():
        cleaned[method] = list(dict.fromkeys([
            f for f in features 
            if f != target_col
        ]))

    # 전체 변수 union
    all_features = sorted(set().union(*cleaned.values()))

    # 투표 테이블 생성
    rows = []
    for feature in all_features:
        selected_by = []
        vote_count = 0

        for method, features in cleaned.items():
            is_selected = feature in features
            if is_selected:
                vote_count += 1
                selected_by.append(method)

        rows.append({
            'feature': feature,
            'vote_count': vote_count,
            'selected_by_wrap': feature in cleaned['wrap'],
            'selected_by_mul': feature in cleaned['mul'],
            'selected_by_boruta': feature in cleaned['boruta'],
            'selected_by': ', '.join(selected_by)
        })

    vote_table = pd.DataFrame(rows)

    # 보기 좋게 정렬
    vote_table = vote_table.sort_values(
        by=['vote_count', 'feature'],
        ascending=[False, True]
    ).reset_index(drop=True)

    # 최종 선택 변수
    selected_features = vote_table.loc[
        vote_table['vote_count'] >= vote_threshold,
        'feature'
    ].tolist()

    return selected_features, vote_table


# ============================================================
# #2. 보팅 실행
# ============================================================

selected_features_vote, vote_table = hard_voting_feature_selection(
    wrap_features=wrap,
    mul_features=mul,
    boruta_features=boruta,
    target_col='Risk_Label',
    vote_threshold=2
)

print("=" * 80)
print("최종 선택 변수 개수:", len(selected_features_vote))
print("=" * 80)

for i, col in enumerate(selected_features_vote, 1):
    print(f"{i}. {col}")

print("\n" + "=" * 80)
print("전체 투표 결과")
print("=" * 80)

display(vote_table)


# ============================================================
# #3. 최종 모델링용 컬럼 생성
# ============================================================

final_cols = selected_features_vote + ['Risk_Label']

print("\n최종 사용 컬럼:")
print(final_cols)

최종 선택 변수 개수: 9
1. NASDAQ_return(%)
2. Brent Crude Oil_return(%)
3. Gold Spot_return(%)
4. KOSDAQ_return(%)
5. KOSPI 200 Volume
6. KOSPI 200 lagged_1_return(%)
7. KOSPI 200_OG
8. KOSPI 200_PPO
9. VKOSPI_Change(%)

전체 투표 결과


,feature,vote_count,selected_by_wrap,selected_by_mul,selected_by_boruta,selected_by
0,NASDAQ_return(%),3,True,True,True,"wrap, mul, boruta"
1,Brent Crude Oil_return(%),2,False,True,True,"mul, boruta"
2,Gold Spot_return(%),2,True,True,False,"wrap, mul"
3,KOSDAQ_return(%),2,True,True,False,"wrap, mul"
4,KOSPI 200 Volume,2,True,False,True,"wrap, boruta"
5,KOSPI 200 lagged_1_return(%),2,False,True,True,"mul, boruta"
6,KOSPI 200_OG,2,True,False,True,"wrap, boruta"
7,KOSPI 200_PPO,2,True,True,False,"wrap, mul"
8,VKOSPI_Change(%),2,True,True,False,"wrap, mul"
9,GJR_VaR_5_t1,1,False,False,True,boruta



최종 사용 컬럼:
['NASDAQ_return(%)', 'Brent Crude Oil_return(%)', 'Gold Spot_return(%)', 'KOSDAQ_return(%)', 'KOSPI 200 Volume', 'KOSPI 200 lagged_1_return(%)', 'KOSPI 200_OG', 'KOSPI 200_PPO', 'VKOSPI_Change(%)', 'Risk_Label']


In [5]:
# Date 기준 정렬 후 저장
data["Date"] = pd.to_datetime(data["Date"])
data = data.sort_values("Date").reset_index(drop=True)

# Date + 최종 선택 변수 + Risk_Label 저장
final_cols_with_date = ["Date"] + final_cols

# 중복 제거
final_cols_with_date = list(dict.fromkeys(final_cols_with_date))

# 실제 존재 컬럼 확인
missing_cols = [col for col in final_cols_with_date if col not in data.columns]
assert len(missing_cols) == 0, f"없는 컬럼 있음: {missing_cols}"

data[final_cols_with_date].to_csv(
    r"../../data/processed/data_selected.csv",
    index=False,
    encoding="utf-8-sig"
)

data[final_cols_with_date].head()

,Date,NASDAQ_return(%),Brent Crude Oil_return(%),Gold Spot_return(%),KOSDAQ_return(%),KOSPI 200 Volume,KOSPI 200 lagged_1_return(%),KOSPI 200_OG,KOSPI 200_PPO,VKOSPI_Change(%),Risk_Label
0,2009-04-17,0.157320,0.0,-1.362586,-2.796416,251600000.0,0.303254,1.339314,3.548332,-2.10,0
1,2009-04-20,-3.953850,0.0,2.234472,1.668519,183300000.0,-0.233192,0.712077,3.485846,1.86,0
2,2009-04-21,2.191930,0.0,-0.553958,1.061549,204200000.0,0.564563,-2.159026,3.379426,0.69,0
3,2009-04-22,0.137996,0.0,1.093648,2.524236,291400000.0,-0.197523,0.767616,3.371206,-3.82,0
4,2009-04-23,0.369276,0.0,1.568707,0.818378,277400000.0,1.408955,0.848630,3.405654,-4.63,0


In [6]:
# ============================================================
# #4. Train/Valid/Test 클래스 비율 확인
# ============================================================

def summarize_ratio(name, labels):
    counts = labels.value_counts().sort_index()
    total = len(labels)
    ratio = (counts / total * 100).round(2)
    print(f"{name}: {total} samples")
    for cls, cnt in counts.items():
        print(f"  Class {cls}: {cnt} ({ratio.loc[cls]}%)")
    print()


def encode_risk_label(y_raw):
    if y_raw.dtype == object:
        y = (
            y_raw.astype(str)
            .str.strip()
            .str.lower()
            .str.replace(r"[\s_\-]+", " ", regex=True)
            .map({"low risk": 0, "high risk": 1, "0": 0, "1": 1})
        )
    else:
        y = pd.to_numeric(y_raw, errors="coerce")

    if y.isna().any():
        raise ValueError("Risk_Label에 0/1로 변환되지 않은 값이 있음")

    return y.astype(int)

selected_data = data[final_cols_with_date].copy()
selected_data["Date"] = pd.to_datetime(selected_data["Date"])
selected_data = selected_data.sort_values("Date").reset_index(drop=True)
selected_data["Risk_Label"] = encode_risk_label(selected_data["Risk_Label"]).values

summarize_ratio("All Data", selected_data["Risk_Label"])

n = len(selected_data)
train_df = selected_data[: int(n * 0.45)].copy()
valid_df = selected_data[int(n * 0.45): int(n * 0.8)].copy()
test_df = selected_data[int(n * 0.8):].copy()

summarize_ratio("Train", train_df["Risk_Label"])
summarize_ratio("Valid", valid_df["Risk_Label"])
summarize_ratio("Test", test_df["Risk_Label"])

sep = int(len(valid_df) * 0.65)
valid_tune = valid_df.iloc[:sep].copy()
valid_final = valid_df.iloc[sep:].copy()

print("=" * 80)
print("Valid set을 0.65:0.35로 나누었을 때")
print("=" * 80)
summarize_ratio("Valid Tune", valid_tune["Risk_Label"])
summarize_ratio("Valid Final", valid_final["Risk_Label"])


All Data: 4108 samples
  Class 0: 3672 (89.39%)
  Class 1: 436 (10.61%)

Train: 1848 samples
  Class 0: 1665 (90.1%)
  Class 1: 183 (9.9%)

Valid: 1438 samples
  Class 0: 1276 (88.73%)
  Class 1: 162 (11.27%)

Test: 822 samples
  Class 0: 731 (88.93%)
  Class 1: 91 (11.07%)

Valid set을 0.65:0.35로 나누었을 때
Valid Tune: 934 samples
  Class 0: 842 (90.15%)
  Class 1: 92 (9.85%)

Valid Final: 504 samples
  Class 0: 434 (86.11%)
  Class 1: 70 (13.89%)

